# Tratamento da Base de Execução Orçamentária — 2024 e 2025

Este notebook executa o ETL da base de execução orçamentária do Município de São Paulo:

1. extrai os arquivos de 2024 e 2025 por meio de `core.downloads.orcamento.load_orcamento`;
2. valida o esquema de cada exercício;
3. preserva as regras de conversão das colunas monetárias;
4. consolida os dois períodos;
5. exporta e relê o CSV final para validar a carga.

A saída mantém exatamente o layout definido na issue #103.


In [ ]:
import pandas as pd
from os import makedirs, path

from core.downloads.orcamento import load_orcamento


In [ ]:
YEARS = [2024, 2025]

OUTPUT_DIR = path.join("data_output", "orcamento")
OUTPUT_FILENAME = "orcamento_2024_2025.csv"
OUTPUT_PATH = path.join(OUTPUT_DIR, OUTPUT_FILENAME)


In [ ]:
EXPECTED_COLUMNS = [
    "Cd_AnoExecucao",
    "Cd_Exercicio",
    "Cd_Dotacao_Id",
    "Administracao",
    "Cd_Orgao",
    "Sigla_Orgao",
    "Ds_Orgao",
    "Cd_Funcao",
    "Ds_Funcao",
    "Cd_SubFuncao",
    "Ds_SubFuncao",
    "Cd_Programa",
    "Ds_Programa",
    "ProjetoAtividade",
    "Ds_Projeto_Atividade",
    "Vl_Orcado_Ano",
    "Vl_Suplementado",
    "Vl_Reduzido",
    "Vl_SuplementadoLiquido",
    "Vl_Orcado_Atualizado",
    "Vl_ReservadoLiquido",
    "Vl_EmpenhadoLiquido",
    "Vl_Liquidado",
    "Vl_Pago",
]

VALUE_COLUMNS = [
    column for column in EXPECTED_COLUMNS
    if column.startswith("Vl_")
]


In [ ]:
ZERO_TOKENS = {
    "",
    "-",
    "--",
    "---",
    "NA",
    "N/A",
    "NAN",
    "NULL",
    "NONE",
}


def parse_numeric_column(series: pd.Series) -> pd.Series:
    """Converte valores monetários textuais em números e falha em valores inválidos."""
    normalized = series.fillna("").astype(str).str.strip()
    normalized = normalized.str.replace("R$", "", regex=False)
    normalized = normalized.str.replace(" ", "", regex=False)
    normalized = normalized.str.replace("\u00A0", "", regex=False)

    has_thousands_and_decimal = (
        normalized.str.contains(".", regex=False)
        & normalized.str.contains(",", regex=False)
    )
    normalized.loc[has_thousands_and_decimal] = (
        normalized.loc[has_thousands_and_decimal]
        .str.replace(".", "", regex=False)
    )

    normalized = normalized.str.replace(",", ".", regex=False)

    zero_mask = normalized.str.upper().isin(ZERO_TOKENS)
    normalized.loc[zero_mask] = "0"

    return pd.to_numeric(normalized)


In [ ]:
raw_dataframes = {}

for year in YEARS:
    dataframe = load_orcamento(year)
    raw_dataframes[year] = dataframe

    print(
        f"Exercício {year}: "
        f"{dataframe.shape[0]} linhas e "
        f"{dataframe.shape[1]} colunas extraídas."
    )


In [ ]:
treated_dataframes = []
rows_by_year = {}

for year, dataframe in raw_dataframes.items():
    missing_columns = [
        column
        for column in EXPECTED_COLUMNS
        if column not in dataframe.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Exercício {year}: colunas ausentes: {missing_columns}"
        )

    rows_before = dataframe.shape[0]
    treated = dataframe[EXPECTED_COLUMNS].copy()

    for column in VALUE_COLUMNS:
        treated[column] = parse_numeric_column(treated[column])

    assert treated.shape[0] == rows_before, (
        f"Exercício {year}: quantidade de linhas alterada no tratamento."
    )
    assert treated.columns.tolist() == EXPECTED_COLUMNS, (
        f"Exercício {year}: layout final diferente do esperado."
    )

    rows_by_year[year] = treated.shape[0]
    treated_dataframes.append(treated)


In [ ]:
df_treated = pd.concat(
    treated_dataframes,
    ignore_index=True,
)

expected_total_rows = sum(rows_by_year.values())

assert df_treated.shape[0] == expected_total_rows, (
    "A quantidade de linhas do consolidado não corresponde "
    "à soma dos exercícios."
)
assert df_treated.columns.tolist() == EXPECTED_COLUMNS, (
    "O consolidado não possui exatamente o layout esperado."
)

duplicate_count = int(df_treated.duplicated().sum())
print(f"Registros completamente duplicados no consolidado: {duplicate_count}")


In [ ]:
exercise_summary = (
    df_treated
    .groupby("Cd_Exercicio", dropna=False)
    .agg(
        quantidade_registros=("Cd_Dotacao_Id", "count"),
        total_orcado=("Vl_Orcado_Ano", "sum"),
        total_empenhado=("Vl_EmpenhadoLiquido", "sum"),
        total_liquidado=("Vl_Liquidado", "sum"),
        total_pago=("Vl_Pago", "sum"),
    )
    .reset_index()
)

exercise_summary


In [ ]:
makedirs(OUTPUT_DIR, exist_ok=True)

df_treated.to_csv(
    OUTPUT_PATH,
    index=False,
    sep=";",
    decimal=",",
    encoding="latin1",
)

print(f"Arquivo exportado para: {OUTPUT_PATH}")


In [ ]:
df_check = pd.read_csv(
    OUTPUT_PATH,
    sep=";",
    decimal=",",
    encoding="latin1",
)

assert df_check.shape[0] == df_treated.shape[0], (
    "Quantidade de linhas inconsistente após a exportação."
)
assert df_check.columns.tolist() == EXPECTED_COLUMNS, (
    "Layout inconsistente após a exportação."
)

print(
    "Carga validada com sucesso: "
    f"{df_check.shape[0]} linhas e "
    f"{df_check.shape[1]} colunas."
)
